In [ ]:


# ========================
# 1. IMPORT LIBRARIES
# ========================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor


# ========================
# 2. UPLOAD DATASET
# ========================
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("expected_ctc.csv")
print("Dataset Shape:", df.shape)
print(df.head())


# ========================
# 3. DATA VISUALIZATION
# ========================

# Boxplot of Salary
plt.figure(figsize=(8,4))
plt.boxplot(df['Expected_CTC'], vert=False)
plt.xlabel('Expected_CTC')
plt.title('Box Plot of Expected_CTC')
plt.show()

# Correlation Heatmap
corr = df.corr(numeric_only=True)

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

print("\nCorrelation with Expected_CTC:")
print(corr['Expected_CTC'].sort_values(ascending=False))


# ========================
# 4. SPLIT FEATURES & TARGET
# ========================
X = df.drop(columns=['Expected_CTC'])
y = df['Expected_CTC']

num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns


# ========================
# 5. PREPROCESSING PIPELINE
# ========================

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])


# ========================
# 6. MODEL PIPELINE
# ========================
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(
        max_depth=10,
        random_state=42
    ))
])


# ========================
# 7. TRAIN TEST SPLIT
# ========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# ========================
# 8. TRAIN MODEL
# ========================
model.fit(X_train, y_train)
print("Training Completed Successfully ")


# ========================
# 9. MODEL EVALUATION
# ========================
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nModel Performance:")
print("MAE  :", round(mae, 2))
print("RMSE :", round(rmse, 2))
print("R2   :", round(r2, 4))


# ========================
# 10. CROSS VALIDATION
# ========================
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')

print("\nCross Validation R2 Scores:", cv_scores)
print("Average CV R2:", cv_scores.mean())


# ========================
# 11. SAMPLE PREDICTIONS (LPA)
# ========================
y_pred_lpa = y_pred / 100000

print("\nSample Predicted Salaries (LPA):")
for salary in y_pred_lpa[:10]:
    print(f"{salary:.2f} LPA")


# ========================
# 12. PREDICT SINGLE CANDIDATE
# ========================
sample_candidate = X_test.iloc[[0]]
predicted_salary = model.predict(sample_candidate)[0] / 100000

print("\nPredicted Salary for Sample Candidate:")
print(f"{predicted_salary:.2f} LPA")


# ========================
# 13. SAVE MODEL
# ========================
joblib.dump(model, "salary_prediction_model.pkl")
print("\nModel Saved Successfully ")


# ========================
# 14. LOAD & PREDICT NEW DATA
# ========================
loaded_model = joblib.load("salary_prediction_model.pkl")

new_data = pd.read_csv("expected_ctc.csv")
predictions = loaded_model.predict(new_data)
predictions_lpa = predictions / 100000

print("\nPredicted Salaries (LPA) for Full Dataset:")
for salary in predictions_lpa[:10]:
    print(f"{salary:.2f} LPA")

KeyboardInterrupt: 